# Ud Din, Lee, Brennecka & Gorai (2024) — point defects and oxygen in wurtzite Al(1-x)Sc(x)N

N. Ud Din, C.-W. Lee, G. L. Brennecka, P. Gorai,
*"Defects and oxygen impurities in ferroelectric wurtzite Al(1-x)Sc(x)N alloys,"*
**Appl. Phys. Lett. 124, 162901 (2024)**,
doi:[10.1063/5.0202097](https://doi.org/10.1063/5.0202097); preprint arXiv:2308.14310.

Companion to `Hirata2020_AlScN_CALPHAD_v2.ipynb`. **Runs on the Python standard library alone.**
`matplotlib` is optional and only used for the plots at the end.

---

## Why this notebook exists

The paper's Figures 1 and S4-S6 plot only the **lower envelope** of the defect formation energies,
i.e. the most stable charge state at each Fermi energy. At a finite temperature every charge state
is populated with its own Boltzmann weight, including where it is not the ground state, so a
concentration model needs *all* the lines, not the envelope. Because
`dE(D,q; E_F) = dE(D,q; E_F=0) + q * E_F`, each straight envelope segment carries its charge state
in its **slope** and the energy we actually need in its **intercept**. Section 9 turns the envelope
back into the full line set; everything downstream uses it.

## What this notebook establishes

| | |
|---|---|
| **Recovered from the figures** | `dE(D,q; E_F=0)` for `V_Al`, `V_Sc`, `V_N`, `O_N` at x = 0.042, 0.125, 0.250, 0.333, under N-rich and most-N-poor chemical potentials, with and without oxygen — 16 panels, 244 charge-state lines |
| **Reproduced exactly** | Figure 2(a): the `V_N` concentration over the whole 500-1000 K range, all four compositions, both growth conditions, to **+0.004 to +0.019 decades** (a 1-4 % error in concentration) with no adjustable parameter |
| **Reproduced exactly** | Figure 2(c) under N-rich conditions, to **0.012-0.121 decades**, using charge neutrality alone and no density-of-states assumption |
| **Not reproducible** | Figure 2(b): the plotted `O_N` concentration is **24.2 +- 1.4 times lower** than the paper's own Eq. (3) applied to the `O_N` energies of its own Figures 1(c,d)/S4-S6(c,d) — a composition- and chemical-potential-independent factor equal, within our uncertainty, to the 24 N sites of the 48-atom SQS cell (section 14) |
| **Not reproducible** | The equilibrium Fermi energy under N-rich, oxygen-free conditions, where it is set by the ab-initio density of states rather than by defects; a parabolic-band model misses it by up to 0.24 eV (section 12) |

## Two headline numbers

1. **The extraction is exact, not approximate.** The figures are vector art, so the envelope segments
   are literal line segments. Fitting a slope to each gives an integer charge state with a residual of
   at most **5.6e-4** across all 244 segments; the reconstructed envelope reproduces the drawn one to
   **0.031 eV** worst case, and the charge-transition levels implied by the fitted intercepts agree
   with the *independently drawn* Figure 3 bars to **0.020 eV** worst case.
2. **Site multiplicity, not chemistry, is what the `O_N` mismatch looks like.** Using
   `N_sites(V_Sc) = x * N_cation` and `N_sites(V_Al) = (1-x) * N_cation` rather than the full cation
   density is what turns a 1.39-decade error in the x = 0.042 net carrier concentration into a
   0.012-decade agreement — and the residual `O_N` factor of 24.6 is the same kind of quantity.


## 2. Conventions

- **Composition** is the cation fraction x in Al(1-x)Sc(x)N. The paper's four compositions correspond
  to 1, 3, 6 and 8 Sc atoms on the 24 cation sites of a 48-atom special quasirandom structure, i.e.
  x = 1/24, 3/24, 6/24, 8/24 = 0.0417, 0.125, 0.250, 0.333.
- **Fermi energy** `E_F` is referenced to the valence band maximum, so 0 <= E_F <= E_gap.
- **Formation energy** `dE(D,q; E_F)` is the paper's *effective* formation energy: the Zhang average
  (SI Eq. 1) over the minimum-energy and maximum-energy Wyckoff sites of the alloy. Everything below
  inherits that definition; we never re-derive it.
- `dE(D,q; E_F) = dE(D,q; E_F=0) + q * E_F` exactly, so a single number per charge state
  (`energy at the valence band maximum`) defines the whole line.
- **Concentration** follows the paper's SI Eq. (3), `C_D = N_sites * exp(-dE_eff / kT)`, with
  **no degeneracy factor** — the paper does not use one, so neither do we.
- **Chemical-potential conditions** are the vertices of the paper's own stability regions: "N-rich"
  is V4 (Delta-mu_N = 0) and "most N-poor" is V1 (Delta-mu_N = -3.174 eV at x = 0.333) of Table S5
  for the oxygen-free panels, and of Table S9 for the oxygen-containing panels.
- All energies are in eV, all concentrations in cm^-3, all temperatures in K.


In [ ]:
import math

BOLTZMANN_CONSTANT_ELECTRONVOLT_PER_KELVIN = 8.617333262e-5
REFERENCE_TEMPERATURE_KELVIN = 1000.0          # the paper solves charge neutrality at 1000 K
CATION_SITES_PER_SUPERCELL = 24                # 48-atom SQS: 24 cations + 24 anions
SCANDIUM_ATOMS_PER_SUPERCELL = {0.042: 1, 0.125: 3, 0.25: 6, 0.333: 8}

COMPOSITIONS = [0.042, 0.125, 0.25, 0.333]
CONDITIONS = ["N-rich, oxygen absent", "most N-poor, oxygen absent",
              "N-rich, oxygen present", "most N-poor, oxygen present"]
DEFECTS = ["V_Al", "V_Sc", "V_N", "O_N"]

print("compositions:", COMPOSITIONS)
print("nominal x from the supercell:",
      [round(SCANDIUM_ATOMS_PER_SUPERCELL[c] / CATION_SITES_PER_SUPERCELL, 4)
       for c in COMPOSITIONS])


## 3. Cross-check harness

Every quantity that is **both** recovered from one figure **and** independently stated in another
figure, a supplementary table, or the running text is recomputed and asserted. Checks that are
**expected to fail** carry `expect_failure=True`: they are the reproducibility gaps of section 14,
and a clean run still ends in `ALL AS EXPECTED`.


In [ ]:
CHECK_RESULTS = []


def check_value(computed, expected, tolerance_relative, label,
                expect_failure=False, note=""):
    """Assert computed ~= expected to a relative tolerance; record PASS/FAIL."""
    if expected == 0:
        relative_error = abs(computed)
    else:
        relative_error = abs(computed - expected) / abs(expected)
    passed = relative_error <= tolerance_relative
    as_expected = (passed != expect_failure)
    CHECK_RESULTS.append({"label": label, "computed": computed, "expected": expected,
                          "relative_error": relative_error, "passed": passed,
                          "expect_failure": expect_failure, "as_expected": as_expected,
                          "note": note})
    tag = "PASS" if passed else "FAIL"
    if expect_failure:
        tag += " (expected FAIL)" if not passed else " (UNEXPECTED PASS)"
    print(f"[{tag:22s}] {label}\n"
          f"{'':24s} computed={computed:.6g}  expected={expected:.6g}  rel.err={relative_error:.3g}"
          + (f"\n{'':24s} {note}" if note else ""))
    return passed


def check_zero(computed, absolute_tolerance, label, expect_failure=False, note=""):
    """Assert |computed| <= tolerance; record PASS/FAIL."""
    passed = abs(computed) <= absolute_tolerance
    as_expected = (passed != expect_failure)
    CHECK_RESULTS.append({"label": label, "computed": computed, "expected": 0.0,
                          "relative_error": abs(computed), "passed": passed,
                          "expect_failure": expect_failure, "as_expected": as_expected,
                          "note": note})
    tag = "PASS" if passed else "FAIL"
    if expect_failure:
        tag += " (expected FAIL)" if not passed else " (UNEXPECTED PASS)"
    print(f"[{tag:22s}] {label}\n"
          f"{'':24s} computed={computed:.6g}  |tol|={absolute_tolerance:.3g}"
          + (f"\n{'':24s} {note}" if note else ""))
    return passed


def summarise_checks():
    total = len(CHECK_RESULTS)
    as_expected = sum(1 for result in CHECK_RESULTS if result["as_expected"])
    print(f"\n{'='*78}\n{as_expected}/{total} checks behaved as expected.")
    unexpected = [result for result in CHECK_RESULTS if not result["as_expected"]]
    if unexpected:
        print("UNEXPECTED:")
        for result in unexpected:
            print(f"  - {result['label']}  (deviation {result['relative_error']:.3g})")
    else:
        deliberate = sum(1 for result in CHECK_RESULTS if result["expect_failure"])
        print(f"ALL AS EXPECTED (including the {deliberate} deliberate failures of section 14).")


## 4. How the figure data was recovered

The published figures are **vector** PDF content, not raster images, so nothing here is digitized in
the usual sense. Each defect curve is drawn as a set of two-point straight segments; each panel's
axes are drawn as a three-point spine polyline plus short tick segments. Reading the content stream
therefore recovers the *exact* device coordinates of every vertex and every tick.

| figure | PDF | Form XObject | what it holds |
|---|---|---|---|
| Fig. 1 | main | 113 (`alscn-defects.pdf`) | dE vs E_F at x = 0.333, 4 panels |
| Fig. 2 | main | 140 (`concentrations.pdf`) | V_N, O_N and net carrier concentrations vs T |
| Fig. 3 | main | 141 (`ctl.pdf`) | charge transition levels and band gaps |
| Fig. S4 | supplementary | 254 (`defects-x042.pdf`) | dE vs E_F at x = 0.042 |
| Fig. S5 | supplementary | 262 (`defects-x125.pdf`) | dE vs E_F at x = 0.125 |
| Fig. S6 | supplementary | 270 (`defects-x250.pdf`) | dE vs E_F at x = 0.250 |

**Calibration.** Each panel is calibrated on its own tick marks: the x ticks are 0-5 eV, the y ticks
are 0, 2, 4, 6, 8 eV. Tick spacings are uniform to better than 0.005 pt within every panel, and
E_F = 0 coincides with the panel's left spine to within 0.05 pt in 15 of the 16 panels. The single
exception is Fig. S6(b), whose six x tick marks are emitted 1.43 pt to the right of the spine and
0.97 % too closely spaced; its x scale was recovered instead from the requirement that the segment
slopes be integers, which then reproduces the same 5.188 eV axis span as its three sibling panels and
the same charge-transition levels as Fig. S6(d). That is recorded here rather than hidden.

**Colour to defect.** The assignment was not inferred from curve positions: the in-plot text labels
`V`+`Al`, `V`+`Sc`, `V`+`N`, `O`+`N` are themselves drawn in the *same* fill colour as their curves,
so the mapping blue -> V_Al, purple -> V_Sc, green -> V_N, red -> O_N is read directly off the glyph
colours. Colours repeat with small rounding differences (0.596 vs 0.592) and were clustered with a
tolerance of 0.03.

**What could not be recovered.** A charge state that never reaches the lower envelope leaves no
segment. The SI states that V_Al and V_Sc were computed in q = -3, -2, -1, 0, +1 and V_N in
q = -1, 0, +1, +2, +3; at x = 0.333 the V_Al q = -1 line is absent from the envelope (a negative-U
0/-2 transition), so only a **lower bound** on its energy is available — section 9 computes it.
Likewise the V_N q = +3 and O_N q = -1 lines are absent wherever their transition would fall outside
the plotted gap.


## 5. Band gaps

The x axis of every dE-vs-E_F panel runs from the valence band maximum to the conduction band
minimum, so **the width of the panel is the band gap**. Figure 3 states the same gaps as text labels
and draws them as the separation of its VBM and CBM bars. Three independent readings.


In [ ]:
# --- band gap, three independent readings -----------------------------------
BAND_GAP_FROM_PANEL_WIDTH = {
    0.042: 5.9239,   # +-0.0001 as read from Figs. S4(a-d); spread over the four panels 0.0001
    0.125: 5.6559,   # +-0.0001 as read from Figs. S5(a-d); spread over the four panels 0.0001
    0.25: 5.1880,    # +-0.0002 as read from Figs. S6(a-d); spread over the four panels 0.0002
    0.333: 5.0400,   # +-0.0001 as read from Figs. 1(a-d);  spread over the four panels 0.0000
}
BAND_GAP_FROM_FIGURE_3_BARS = {
    0.042: 5.9240,   # +-0.0005 as read from Fig. 3, CBM bar minus VBM bar
    0.125: 5.6560,   # +-0.0005 as read from Fig. 3
    0.25: 5.1880,    # +-0.0005 as read from Fig. 3
    0.333: 5.0400,   # +-0.0005 as read from Fig. 3
}
BAND_GAP_PRINTED_IN_FIGURE_3 = {0.042: 5.92, 0.125: 5.66, 0.25: 5.19, 0.333: 5.04}
BAND_GAP_ALUMINIUM_NITRIDE_GW = 6.11   # stated in the main text

print(f"{'x':>6} {'panel width':>12} {'Fig. 3 bars':>12} {'Fig. 3 text':>12} {'max spread':>11}")
for composition in COMPOSITIONS:
    readings = [BAND_GAP_FROM_PANEL_WIDTH[composition],
                BAND_GAP_FROM_FIGURE_3_BARS[composition],
                BAND_GAP_PRINTED_IN_FIGURE_3[composition]]
    print(f"{composition:6.3f} {readings[0]:12.4f} {readings[1]:12.4f} {readings[2]:12.2f} "
          f"{max(readings) - min(readings):11.4f}")

BAND_GAP = dict(BAND_GAP_FROM_PANEL_WIDTH)


## 6. Defect formation energies at the valence band maximum

`dE(D,q; E_F=0)` in eV, i.e. the **intercept** of each charge state's line. The lines themselves are
`dE(D,q; E_F=0) + q * E_F` for `0 <= E_F <= E_gap` — that is the whole point of section 9.

Uncertainty on every entry is **+-0.02 eV**, set not by the calibration (which is exact to ~0.0002 eV)
but by the figures' own vertex rendering: neighbouring segments of the same polyline are drawn with
end points that disagree by up to 0.6 pt, so an intercept averaged over one segment's two ends
inherits about 0.012-0.020 eV of that. Section 8 measures it.


In [ ]:
# ---------------------------------------------------------------------------
# dE(D, q; E_F = 0) in eV, +-0.02 eV, read from the lower-envelope segments of
# Fig. 1 (x = 0.333) and Figs. S4/S5/S6 (x = 0.042 / 0.125 / 0.250).
# Keys are (composition, chemical-potential condition); the inner keys are charge states.
# ---------------------------------------------------------------------------
FORMATION_ENERGY_AT_VALENCE_BAND_MAXIMUM = {
    (0.042, "N-rich, oxygen absent"): {
        "V_Al": {+1: 5.894, +0: 6.513, -1: 8.362, -2: 10.525, -3: 13.098},
        "V_Sc": {+1: 3.315, +0: 4.321, -1: 6.332, -2: 8.692, -3: 11.622},
        "V_N": {+3: -1.343, +2: -0.248, +1: 1.181, +0: 4.927, -1: 9.280},
    },  # Fig. S4(a)
    (0.042, "most N-poor, oxygen absent"): {
        "V_Al": {+1: 9.065, +0: 9.685, -1: 11.533, -2: 13.697, -3: 16.269},
        "V_Sc": {+1: 7.355, +0: 8.361, -1: 10.372, -2: 12.732, -3: 15.662},
        "V_N": {+1: -2.027, +0: 1.719, -1: 6.073},
    },  # Fig. S4(b)
    (0.042, "N-rich, oxygen present"): {
        "V_Al": {+1: 5.858, +0: 6.478, -1: 8.326, -2: 10.489, -3: 13.062},
        "V_Sc": {+1: 4.147, +0: 5.153, -1: 7.164, -2: 9.524, -3: 12.454},
        "V_N": {+3: -1.346, +2: -0.251, +1: 1.178, +0: 4.924, -1: 9.277},
        "O_N": {+1: -2.023, +0: 3.705},
    },  # Fig. S4(c)
    (0.042, "most N-poor, oxygen present"): {
        "V_Al": {+1: 9.065, +0: 9.685, -1: 11.533, -2: 13.697, -3: 16.270},
        "V_Sc": {+1: 6.873, +0: 7.880, -1: 9.891, -2: 12.251, -3: 15.181},
        "V_N": {+1: -2.042, +0: 1.704, -1: 6.057},
        "O_N": {+1: -3.377, +0: 2.351},
    },  # Fig. S4(d)
    (0.125, "N-rich, oxygen absent"): {
        "V_Al": {+1: 5.223, +0: 6.183, -1: 7.837, -2: 10.032, -3: 12.591},
        "V_Sc": {+1: 3.929, +0: 4.866, -1: 6.734, -2: 9.056, -3: 11.989},
        "V_N": {+3: -1.556, +2: -0.623, +1: 0.910, +0: 4.312, -1: 8.320},
    },  # Fig. S5(a)
    (0.125, "most N-poor, oxygen absent"): {
        "V_Al": {+1: 8.394, +0: 9.354, -1: 11.009, -2: 13.204, -3: 15.763},
        "V_Sc": {+1: 7.503, +0: 8.440, -1: 10.308, -2: 12.630, -3: 15.563},
        "V_N": {+1: -2.312, +0: 1.091, -1: 5.098},
    },  # Fig. S5(b)
    (0.125, "N-rich, oxygen present"): {
        "V_Al": {+1: 5.172, +0: 6.133, -1: 7.787, -2: 9.982, -3: 12.541},
        "V_Sc": {+1: 4.281, +0: 5.218, -1: 7.086, -2: 9.408, -3: 12.341},
        "V_N": {+3: -1.559, +2: -0.626, +1: 0.907, +0: 4.309, -1: 8.317},
        "O_N": {+1: -2.234, +0: 2.003},
    },  # Fig. S5(c)
    (0.125, "most N-poor, oxygen present"): {
        "V_Al": {+1: 8.394, +0: 9.354, -1: 11.009, -2: 13.204, -3: 15.763},
        "V_Sc": {+1: 7.101, +0: 8.038, -1: 9.906, -2: 12.228, -3: 15.161},
        "V_N": {+1: -2.265, +0: 1.137, -1: 5.145},
        "O_N": {+1: -3.556, +0: 0.681},
    },  # Fig. S5(d)
    (0.25, "N-rich, oxygen absent"): {
        "V_Al": {+1: 5.489, +0: 6.299, -1: 7.724, -2: 9.496, -3: 11.568},
        "V_Sc": {+1: 4.182, +0: 4.982, -1: 6.429, -2: 8.206, -3: 10.386},
        "V_N": {+3: -2.127, +2: -0.637, +1: 1.256, +0: 4.316, -1: 8.092},
    },  # Fig. S6(a)
    (0.25, "most N-poor, oxygen absent"): {
        "V_Al": {+1: 8.661, +0: 9.471, -1: 10.896, -2: 12.668, -3: 14.739},
        "V_Sc": {+1: 7.471, +0: 8.271, -1: 9.718, -2: 11.495, -3: 13.675},
        "V_N": {+2: -3.838, +1: -1.945, +0: 1.115, -1: 4.891},
    },  # Fig. S6(b)
    (0.25, "N-rich, oxygen present"): {
        "V_Al": {+1: 5.460, +0: 6.271, -1: 7.695, -2: 9.467, -3: 11.539},
        "V_Sc": {+1: 4.270, +0: 5.070, -1: 6.517, -2: 8.294, -3: 10.474},
        "V_N": {+3: -2.130, +2: -0.641, +1: 1.252, +0: 4.313, -1: 8.089},
        "O_N": {+1: -1.960, +0: 1.641, -1: 5.889},
    },  # Fig. S6(c)
    (0.25, "most N-poor, oxygen present"): {
        "V_Al": {+1: 8.661, +0: 9.471, -1: 10.896, -2: 12.668, -3: 14.739},
        "V_Sc": {+1: 7.354, +0: 8.154, -1: 9.601, -2: 11.378, -3: 13.558},
        "V_N": {+2: -3.812, +1: -1.920, +0: 1.141, -1: 4.917},
        "O_N": {+1: -3.076, +0: 0.525, -1: 4.772},
    },  # Fig. S6(d)
    (0.333, "N-rich, oxygen absent"): {
        "V_Al": {+1: 5.614, +0: 5.977, -2: 8.675, -3: 10.563},
        "V_Sc": {+1: 4.892, +0: 5.731, -1: 7.387, -2: 9.122, -3: 11.182},
        "V_N": {+3: -2.207, +2: -0.619, +1: 1.190, +0: 4.430, -1: 8.160},
    },  # Fig. 1(a)
    (0.333, "most N-poor, oxygen absent"): {
        "V_Al": {+1: 8.786, +0: 9.149, -2: 11.846, -3: 13.762},
        "V_Sc": {+1: 8.069, +0: 8.909, -1: 10.565, -2: 12.299, -3: 14.359},
        "V_N": {+2: -3.794, +1: -1.983, +0: 1.257, -1: 4.986},
    },  # Fig. 1(b)
    (0.333, "N-rich, oxygen present"): {
        "V_Al": {+1: 5.617, +0: 5.980, -2: 8.677, -3: 10.565},
        "V_Sc": {+1: 4.895, +0: 5.734, -1: 7.389, -2: 9.125, -3: 11.184},
        "V_N": {+3: -2.208, +2: -0.621, +1: 1.189, +0: 4.429, -1: 8.159},
        "O_N": {+1: -1.991, +0: 1.817, -1: 5.954},
    },  # Fig. 1(c)
    (0.333, "most N-poor, oxygen present"): {
        "V_Al": {+1: 8.789, +0: 9.152, -2: 11.849, -3: 13.737},
        "V_Sc": {+1: 8.066, +0: 8.906, -1: 10.562, -2: 12.296, -3: 14.356},
        "V_N": {+2: -3.792, +1: -1.982, +0: 1.258, -1: 4.988},
        "O_N": {+1: -3.048, +0: 0.760, -1: 4.897},
    },  # Fig. 1(d)
}

segment_count = sum(len(states) for panel in FORMATION_ENERGY_AT_VALENCE_BAND_MAXIMUM.values()
                    for states in panel.values())
print(f"{len(FORMATION_ENERGY_AT_VALENCE_BAND_MAXIMUM)} panels, "
      f"{segment_count} charge-state lines recovered")


In [ ]:
# Fitted slope minus the nearest integer, one entry per line above, same ordering.
# A perfect extraction gives exactly zero; these are the actual residuals.
SLOPE_RESIDUALS = {
    (0.042, "N-rich, oxygen absent"): {"V_Al": [-0.00012, +0.00000, +0.00002, -0.00011, -0.00001], "V_Sc": [-0.00005, +0.00000, +0.00024, -0.00018, +0.00016], "V_N": [+0.00011, -0.00007, +0.00003, +0.00000, -0.00007]},
    (0.042, "most N-poor, oxygen absent"): {"V_Al": [+0.00005, +0.00000, +0.00027, -0.00007, +0.00000], "V_Sc": [+0.00008, +0.00000, -0.00022, +0.00016, +0.00000], "V_N": [-0.00000, +0.00000, -0.00004]},
    (0.042, "N-rich, oxygen present"): {"V_Al": [+0.00025, +0.00000, +0.00013, -0.00016, +0.00002], "V_Sc": [+0.00010, +0.00000, -0.00009, -0.00007, +0.00007], "V_N": [+0.00004, +0.00006, -0.00001, +0.00000, +0.00005], "O_N": [-0.00002, +0.00000]},
    (0.042, "most N-poor, oxygen present"): {"V_Al": [-0.00004, +0.00000, -0.00012, +0.00056, -0.00005], "V_Sc": [+0.00005, +0.00000, +0.00017, +0.00005, -0.00001], "V_N": [-0.00002, +0.00000, -0.00003], "O_N": [-0.00001, +0.00000]},
    (0.125, "N-rich, oxygen absent"): {"V_Al": [-0.00006, +0.00000, +0.00016, -0.00047, +0.00010], "V_Sc": [+0.00009, +0.00000, -0.00019, -0.00018, +0.00000], "V_N": [+0.00009, -0.00024, +0.00001, +0.00000, -0.00004]},
    (0.125, "most N-poor, oxygen absent"): {"V_Al": [-0.00009, +0.00000, +0.00004, -0.00009, -0.00001], "V_Sc": [+0.00003, +0.00000, +0.00009, -0.00005, +0.00002], "V_N": [+0.00001, +0.00000, -0.00001]},
    (0.125, "N-rich, oxygen present"): {"V_Al": [-0.00000, +0.00000, +0.00004, -0.00003, -0.00002], "V_Sc": [+0.00004, +0.00000, +0.00000, -0.00020, +0.00007], "V_N": [-0.00017, +0.00006, +0.00002, +0.00000, -0.00003], "O_N": [+0.00002, +0.00000]},
    (0.125, "most N-poor, oxygen present"): {"V_Al": [+0.00006, +0.00000, -0.00006, +0.00028, -0.00002], "V_Sc": [-0.00007, +0.00000, -0.00013, -0.00007, +0.00003], "V_N": [+0.00002, +0.00000, -0.00002], "O_N": [-0.00002, +0.00000]},
    (0.25, "N-rich, oxygen absent"): {"V_Al": [+0.00004, +0.00000, -0.00018, -0.00011, -0.00000], "V_Sc": [+0.00015, +0.00000, +0.00014, -0.00021, +0.00009], "V_N": [+0.00007, -0.00023, +0.00004, +0.00000, +0.00005]},
    (0.25, "most N-poor, oxygen absent"): {"V_Al": [-0.00011, +0.00000, +0.00016, -0.00006, +0.00000], "V_Sc": [+0.00001, +0.00000, -0.00009, +0.00016, -0.00004], "V_N": [-0.00051, +0.00006, +0.00000, -0.00005]},
    (0.25, "N-rich, oxygen present"): {"V_Al": [-0.00008, +0.00000, +0.00015, +0.00001, +0.00003], "V_Sc": [+0.00003, +0.00000, +0.00022, +0.00002, -0.00010], "V_N": [-0.00001, +0.00008, -0.00005, +0.00000, +0.00005], "O_N": [-0.00003, +0.00000, -0.00010]},
    (0.25, "most N-poor, oxygen present"): {"V_Al": [+0.00003, +0.00000, -0.00036, +0.00015, -0.00002], "V_Sc": [+0.00011, +0.00000, -0.00040, +0.00020, +0.00003], "V_N": [+0.00010, -0.00006, +0.00000, +0.00000], "O_N": [-0.00000, +0.00000, +0.00003]},
    (0.333, "N-rich, oxygen absent"): {"V_Al": [-0.00001, +0.00000, -0.00002, +0.00001], "V_Sc": [-0.00001, +0.00000, -0.00018, +0.00008, +0.00000], "V_N": [+0.00003, +0.00002, -0.00001, +0.00000, -0.00000]},
    (0.333, "most N-poor, oxygen absent"): {"V_Al": [-0.00002, +0.00000, -0.00001, +0.00000], "V_Sc": [+0.00004, +0.00000, -0.00018, +0.00009, -0.00001], "V_N": [-0.00020, -0.00001, +0.00000, +0.00000]},
    (0.333, "N-rich, oxygen present"): {"V_Al": [-0.00001, +0.00000, -0.00004, +0.00003], "V_Sc": [-0.00003, +0.00000, +0.00016, -0.00002, -0.00001], "V_N": [+0.00002, +0.00011, +0.00000, +0.00000, +0.00001], "O_N": [-0.00000, +0.00000, +0.00001]},
    (0.333, "most N-poor, oxygen present"): {"V_Al": [-0.00001, +0.00000, -0.00005, -0.00001], "V_Sc": [-0.00002, +0.00000, +0.00015, -0.00003, +0.00001], "V_N": [-0.00006, +0.00000, +0.00000, +0.00001], "O_N": [+0.00000, +0.00000, +0.00001]},
}

# Equilibrium Fermi energy at 1000 K, read from the vertical marker drawn in each panel.
# +-0.002 eV (the marker is a single stroked line; the uncertainty is its calibration).
EQUILIBRIUM_FERMI_ENERGY_AT_1000_KELVIN = {
    (0.042, "N-rich, oxygen absent"): 2.8280,
    (0.042, "most N-poor, oxygen absent"): 4.0399,
    (0.042, "N-rich, oxygen present"): 3.6089,
    (0.042, "most N-poor, oxygen present"): 4.5629,
    (0.125, "N-rich, oxygen absent"): 2.6336,
    (0.125, "most N-poor, oxygen absent"): 3.7050,
    (0.125, "N-rich, oxygen present"): 3.5850,
    (0.125, "most N-poor, oxygen present"): 4.2150,
    (0.25, "N-rich, oxygen absent"): 2.5209,
    (0.25, "most N-poor, oxygen absent"): 3.4180,
    (0.25, "N-rich, oxygen present"): 3.0461,
    (0.25, "most N-poor, oxygen present"): 3.8789,
    (0.333, "N-rich, oxygen absent"): 2.4986,
    (0.333, "most N-poor, oxygen absent"): 3.4829,
    (0.333, "N-rich, oxygen present"): 3.0560,
    (0.333, "most N-poor, oxygen present"): 3.8770,
}

# Fermi energy of each kink in the DRAWN envelope, i.e. the midpoint of the two
# neighbouring segment end points, +-0.02 eV. Independent of the intercepts above:
# these come from vertex positions, those from slopes and offsets.
MEASURED_ENVELOPE_KINKS = {
    (0.042, "N-rich, oxygen absent"): {"V_Al": [0.620, 1.848, 2.164, 2.585], "V_Sc": [1.009, 2.010, 2.360, 2.930], "V_N": [1.096, 1.428, 3.747, 4.352]},
    (0.042, "most N-poor, oxygen absent"): {"V_Al": [0.620, 1.848, 2.164, 2.585], "V_Sc": [1.009, 2.010, 2.360, 2.930], "V_N": [3.747, 4.352]},
    (0.042, "N-rich, oxygen present"): {"V_Al": [0.620, 1.849, 2.163, 2.573], "V_Sc": [0.990, 2.011, 2.360, 2.930], "V_N": [1.095, 1.427, 3.746, 4.353], "O_N": [5.727]},
    (0.042, "most N-poor, oxygen present"): {"V_Al": [0.620, 1.849, 2.163, 2.573], "V_Sc": [0.990, 2.011, 2.360, 2.930], "V_N": [3.746, 4.353], "O_N": [5.727]},
    (0.125, "N-rich, oxygen absent"): {"V_Al": [0.960, 1.655, 2.195, 2.559], "V_Sc": [0.937, 1.868, 2.322, 2.933], "V_N": [0.933, 1.533, 3.403, 4.007]},
    (0.125, "most N-poor, oxygen absent"): {"V_Al": [0.960, 1.655, 2.195, 2.559], "V_Sc": [0.937, 1.868, 2.322, 2.933], "V_N": [3.403, 4.007]},
    (0.125, "N-rich, oxygen present"): {"V_Al": [0.961, 1.654, 2.195, 2.559], "V_Sc": [0.937, 1.868, 2.322, 2.933], "V_N": [0.933, 1.533, 3.402, 4.008], "O_N": [4.237]},
    (0.125, "most N-poor, oxygen present"): {"V_Al": [0.961, 1.654, 2.195, 2.559], "V_Sc": [0.937, 1.868, 2.322, 2.933], "V_N": [3.402, 4.008], "O_N": [4.237]},
    (0.25, "N-rich, oxygen absent"): {"V_Al": [0.810, 1.425, 1.772, 2.071], "V_Sc": [0.800, 1.447, 1.778, 2.180], "V_N": [1.490, 1.892, 3.061, 3.776]},
    (0.25, "most N-poor, oxygen absent"): {"V_Al": [0.810, 1.424, 1.772, 2.071], "V_Sc": [0.800, 1.446, 1.778, 2.180], "V_N": [1.892, 3.060, 3.776]},
    (0.25, "N-rich, oxygen present"): {"V_Al": [0.810, 1.424, 1.772, 2.071], "V_Sc": [0.800, 1.446, 1.778, 2.180], "V_N": [1.489, 1.892, 3.060, 3.776], "O_N": [3.601, 4.247]},
    (0.25, "most N-poor, oxygen present"): {"V_Al": [0.810, 1.425, 1.772, 2.071], "V_Sc": [0.800, 1.447, 1.778, 2.180], "V_N": [1.892, 3.061, 3.776], "O_N": [3.601, 4.247]},
    (0.333, "N-rich, oxygen absent"): {"V_Al": [0.354, 1.361, 1.888], "V_Sc": [0.870, 1.670, 1.735, 2.060], "V_N": [1.587, 1.810, 3.225, 3.750]},
    (0.333, "most N-poor, oxygen absent"): {"V_Al": [0.354, 1.361, 1.888], "V_Sc": [0.870, 1.670, 1.735, 2.060], "V_N": [1.810, 3.225, 3.750]},
    (0.333, "N-rich, oxygen present"): {"V_Al": [0.354, 1.361, 1.888], "V_Sc": [0.860, 1.670, 1.735, 2.059], "V_N": [1.587, 1.810, 3.225, 3.750], "O_N": [3.805, 4.137]},
    (0.333, "most N-poor, oxygen present"): {"V_Al": [0.354, 1.361, 1.888], "V_Sc": [0.860, 1.670, 1.735, 2.059], "V_N": [1.810, 3.225, 3.750], "O_N": [3.805, 4.137]},
}

all_residuals = [value for panel in SLOPE_RESIDUALS.values()
                 for values in panel.values() for value in values]
print(f"{len(all_residuals)} slope residuals, "
      f"max |residual| = {max(abs(v) for v in all_residuals):.2e}")


## 7. Charge transition levels drawn in Figure 3

Figure 3 draws the V_N and O_N charge transition levels as horizontal bars between a VBM and a CBM
bar. It is a **different figure in a different file**, produced from the same underlying numbers, so
it is an independent check on the intercepts of section 6: a transition level `(q1/q2)` must equal
`(dE(q2,0) - dE(q1,0)) / (q1 - q2)`.


In [ ]:
# Charge transition levels in eV above the VBM, +-0.005 eV as read from Fig. 3.
# Only levels that fall inside the gap are drawn, which is why x = 0.042 and 0.125
# carry a single O_N bar: their O_N 0/-1 level lies above the conduction band minimum.
CHARGE_TRANSITION_LEVELS_FIGURE_3 = {
    0.042: {
        "V_N": {"+3/+2": 1.0960, "+2/+1": 1.4280, "+1/0": 3.7470, "0/-1": 4.3520},
        "O_N": {"+1/0": 5.7270},
    },
    0.125: {
        "V_N": {"+3/+2": 0.9330, "+2/+1": 1.5330, "+1/0": 3.4030, "0/-1": 4.0070},
        "O_N": {"+1/0": 4.2370},
    },
    0.25: {
        "V_N": {"+3/+2": 1.4900, "+2/+1": 1.8920, "+1/0": 3.0610, "0/-1": 3.7760},
        "O_N": {"+1/0": 3.6010, "0/-1": 4.2470},
    },
    0.333: {
        "V_N": {"+3/+2": 1.5870, "+2/+1": 1.8100, "+1/0": 3.2240, "0/-1": 3.7500},
        "O_N": {"+1/0": 3.8080, "0/-1": 4.1370},
    },
}

for composition in COMPOSITIONS:
    entry = CHARGE_TRANSITION_LEVELS_FIGURE_3[composition]
    print(f"x = {composition:5.3f}  V_N " +
          "  ".join(f"{name}={value:.3f}" for name, value in entry["V_N"].items()) +
          "   |  O_N " +
          "  ".join(f"{name}={value:.3f}" for name, value in entry["O_N"].items()))


## 8. Extraction quality

Four checks, in increasing order of how hard they are to pass by accident.

1. **Slope integrality.** Nothing forces a fitted slope to be an integer; the physics does.
2. **Envelope reconstruction.** The kinks of `min_q [dE(q,0) + q*E_F]` must land on the kinks of the
   drawn polyline. The two come from different information: the reconstruction from segment *slopes
   and intercepts*, the drawn kinks from *vertex positions*.
3. **Charge-transition levels.** The intercepts of section 6 must reproduce the bars of Figure 3.
4. **Chemical potentials.** The *difference* between the N-rich and most-N-poor panels must equal the
   difference in the elemental chemical potentials tabulated in SI Tables S5 and S9, which were never
   used in the extraction.


In [ ]:
CHECK_RESULTS.clear()

# --- 1. slope integrality --------------------------------------------------
check_zero(max(abs(value) for value in all_residuals), 1.0e-3,
           "Slope integrality: every envelope segment has an integer charge state",
           note=f"{len(all_residuals)} segments; "
                f"root-mean-square residual "
                f"{math.sqrt(sum(v*v for v in all_residuals)/len(all_residuals)):.2e}")


# --- 2. envelope reconstruction --------------------------------------------
def lower_envelope_energy(charge_states, fermi_energy):
    """min over q of dE(q, 0) + q * E_F."""
    return min(energy + charge * fermi_energy for charge, energy in charge_states.items())


def reconstructed_envelope_kinks(charge_states):
    """Fermi energies where min_q[dE(q,0) + q*E_F] changes its minimising charge state."""
    charges = sorted(charge_states, reverse=True)
    return [(charge_states[low] - charge_states[high]) / (high - low)
            for high, low in zip(charges, charges[1:])]


worst_envelope_deviation, worst_envelope_label = 0.0, ""
envelope_deviations = []
for key, panel in FORMATION_ENERGY_AT_VALENCE_BAND_MAXIMUM.items():
    for defect, charge_states in panel.items():
        reconstructed = reconstructed_envelope_kinks(charge_states)
        drawn = MEASURED_ENVELOPE_KINKS[key][defect]
        for reconstructed_kink, drawn_kink in zip(reconstructed, drawn):
            deviation = abs(reconstructed_kink - drawn_kink)
            envelope_deviations.append(deviation)
            if deviation > worst_envelope_deviation:
                worst_envelope_deviation = deviation
                worst_envelope_label = f"x={key[0]} {key[1]} {defect}"
check_zero(worst_envelope_deviation, 0.035,
           "Envelope reconstruction: kinks of min_q[dE(q,0)+q*E_F] land on the drawn kinks",
           note=f"{len(envelope_deviations)} kinks; worst case {worst_envelope_label}; "
                f"mean deviation "
                f"{sum(envelope_deviations)/len(envelope_deviations):.4f} eV")

# --- 3. charge transition levels against Figure 3 --------------------------
TRANSITION_PAIRS = {"+3/+2": (3, 2), "+2/+1": (2, 1), "+1/0": (1, 0), "0/-1": (0, -1)}
worst_transition_error, worst_transition_label = 0.0, ""
for composition in COMPOSITIONS:
    condition_for = {"V_N": "N-rich, oxygen absent", "O_N": "N-rich, oxygen present"}
    for defect, levels in CHARGE_TRANSITION_LEVELS_FIGURE_3[composition].items():
        charge_states = FORMATION_ENERGY_AT_VALENCE_BAND_MAXIMUM[
            (composition, condition_for[defect])][defect]
        for name, (upper_charge, lower_charge) in TRANSITION_PAIRS.items():
            if name not in levels or upper_charge not in charge_states \
                    or lower_charge not in charge_states:
                continue
            computed = ((charge_states[lower_charge] - charge_states[upper_charge])
                        / (upper_charge - lower_charge))
            error = abs(computed - levels[name])
            if error > worst_transition_error:
                worst_transition_error = error
                worst_transition_label = f"x={composition} {defect} {name}"
check_zero(worst_transition_error, 0.025,
           "Charge transition levels from section 6 intercepts vs the Fig. 3 bars",
           note=f"worst case {worst_transition_label}; Fig. 3 is an independently drawn figure")

# --- 4. chemical potentials, SI Tables S5 and S9 ---------------------------
# x = 0.333. Ternary (oxygen absent), Table S5: V1 = most N-poor, V4 = N-rich.
DELTA_MU_TERNARY = {"V1": {"Al": 0.000, "N": -3.174, "Sc": -1.136},
                    "V4": {"Al": -3.174, "N": 0.000, "Sc": -4.310}}
# Quaternary (oxygen present), Table S9.
DELTA_MU_QUATERNARY = {"V1": {"Al": 0.000, "N": -3.174, "O": -5.818, "Sc": -1.136},
                       "V4": {"Al": -3.172, "N": 0.000, "O": -3.699, "Sc": -4.314}}

for defect, element, table in (("V_Al", "Al", DELTA_MU_TERNARY),
                               ("V_Sc", "Sc", DELTA_MU_TERNARY),
                               ("V_N", "N", DELTA_MU_TERNARY)):
    n_rich = FORMATION_ENERGY_AT_VALENCE_BAND_MAXIMUM[
        (0.333, "N-rich, oxygen absent")][defect][0]
    n_poor = FORMATION_ENERGY_AT_VALENCE_BAND_MAXIMUM[
        (0.333, "most N-poor, oxygen absent")][defect][0]
    expected = table["V1"][element] - table["V4"][element]
    check_value(n_poor - n_rich, expected, 0.005,
                f"{defect}: dE(most N-poor) - dE(N-rich) equals Delta-mu_{element} of Table S5")

oxygen_n_rich = FORMATION_ENERGY_AT_VALENCE_BAND_MAXIMUM[
    (0.333, "N-rich, oxygen present")]["O_N"][0]
oxygen_n_poor = FORMATION_ENERGY_AT_VALENCE_BAND_MAXIMUM[
    (0.333, "most N-poor, oxygen present")]["O_N"][0]
expected_oxygen = ((DELTA_MU_QUATERNARY["V1"]["N"] - DELTA_MU_QUATERNARY["V4"]["N"])
                   - (DELTA_MU_QUATERNARY["V1"]["O"] - DELTA_MU_QUATERNARY["V4"]["O"]))
check_value(oxygen_n_poor - oxygen_n_rich, expected_oxygen, 0.005,
            "O_N: dE(most N-poor) - dE(N-rich) equals Delta-mu_N - Delta-mu_O of Table S9")

# --- band gaps: panel width vs Fig. 3 --------------------------------------
for composition in COMPOSITIONS:
    check_value(BAND_GAP_FROM_PANEL_WIDTH[composition],
                BAND_GAP_FROM_FIGURE_3_BARS[composition], 5.0e-4,
                f"x = {composition}: gap from panel width vs Fig. 3 bars")

# --- charge transition levels do not depend on the chemical potential ------
worst_condition_drift = 0.0
for composition in COMPOSITIONS:
    for defect in DEFECTS:
        reference = None
        for condition in CONDITIONS:
            panel = FORMATION_ENERGY_AT_VALENCE_BAND_MAXIMUM[(composition, condition)]
            if defect not in panel:
                continue
            charges = sorted(panel[defect], reverse=True)
            levels = [(panel[defect][low] - panel[defect][high]) / (high - low)
                      for high, low in zip(charges, charges[1:])]
            if reference is None:
                reference = levels
            elif len(reference) == len(levels):
                worst_condition_drift = max(
                    worst_condition_drift,
                    max(abs(a - b) for a, b in zip(reference, levels)))
check_zero(worst_condition_drift, 0.030,
           "Charge transition levels are the same in all four chemical-potential panels",
           note="a transition level is a difference of formation energies at fixed composition, "
                "so the chemical potentials must cancel exactly; the residual is the figures' own "
                "vertex-rendering slop and is the basis of the +-0.02 eV quoted in section 6")


## 9. Every charge state as a full line across the gap

This is what the notebook exists to produce. Each entry of section 6 becomes a straight line
`dE(q, E_F) = dE(q, 0) + q * E_F` defined over the **whole** gap, not just over the interval where it
happens to be the ground state. The lower envelope is recovered as `min_q` of those lines, which is
what the paper plots; everything above the envelope is what the paper's figures throw away and what a
finite-temperature population needs.

Where a charge state never reaches the envelope it leaves no segment, and only a **lower bound** on
its energy can be stated: at the Fermi energy where its two neighbours cross, its own line must lie
above them.


In [ ]:
def formation_energy(composition, condition, defect, charge, fermi_energy):
    """dE(D, q; E_F) in eV."""
    return (FORMATION_ENERGY_AT_VALENCE_BAND_MAXIMUM[(composition, condition)][defect][charge]
            + charge * fermi_energy)


def charge_state_lines(composition, condition, defect, sample_count=200):
    """Every charge state as a complete line over 0 <= E_F <= E_gap."""
    gap = BAND_GAP[composition]
    charge_states = FORMATION_ENERGY_AT_VALENCE_BAND_MAXIMUM[(composition, condition)][defect]
    fermi_energies = [gap * index / sample_count for index in range(sample_count + 1)]
    return fermi_energies, {charge: [energy + charge * value for value in fermi_energies]
                            for charge, energy in charge_states.items()}


def charge_transition_levels(composition, condition, defect):
    """Fermi energies at which the ground-state charge changes, in eV above the VBM."""
    charge_states = FORMATION_ENERGY_AT_VALENCE_BAND_MAXIMUM[(composition, condition)][defect]
    charges = sorted(charge_states, reverse=True)
    return {(high, low): (charge_states[low] - charge_states[high]) / (high - low)
            for high, low in zip(charges, charges[1:])}


def missing_charge_state_lower_bound(composition, condition, defect, charge):
    """Lower bound on dE(q, 0) for a charge state absent from the envelope: at the
    Fermi energy where its two neighbours cross, its line must lie above them."""
    charge_states = FORMATION_ENERGY_AT_VALENCE_BAND_MAXIMUM[(composition, condition)][defect]
    above = min((q for q in charge_states if q > charge), default=None)
    below = max((q for q in charge_states if q < charge), default=None)
    if above is None or below is None:
        return None
    crossing = (charge_states[below] - charge_states[above]) / (above - below)
    return lower_envelope_energy(charge_states, crossing) - charge * crossing


print("Complete charge-state line set, x = 0.333, most N-poor, oxygen present")
print(f"{'defect':6s} {'q':>3} {'dE(q, E_F=0)':>13} {'dE at E_F,eq':>13} {'ground state over':>24}")
condition = "most N-poor, oxygen present"
equilibrium = EQUILIBRIUM_FERMI_ENERGY_AT_1000_KELVIN[(0.333, condition)]
for defect in DEFECTS:
    panel = FORMATION_ENERGY_AT_VALENCE_BAND_MAXIMUM[(0.333, condition)]
    if defect not in panel:
        continue
    charge_states = panel[defect]
    levels = charge_transition_levels(0.333, condition, defect)
    boundaries = [0.0] + [levels[pair] for pair in sorted(levels, reverse=True)] \
        + [BAND_GAP[0.333]]
    for index, charge in enumerate(sorted(charge_states, reverse=True)):
        span = f"{boundaries[index]:.3f} to {boundaries[index+1]:.3f} eV"
        print(f"{defect:6s} {charge:+3d} {charge_states[charge]:13.3f} "
              f"{formation_energy(0.333, condition, defect, charge, equilibrium):13.3f} "
              f"{span:>24}")

bound = missing_charge_state_lower_bound(0.333, "N-rich, oxygen absent", "V_Al", -1)
print(f"\nV_Al q = -1 at x = 0.333 never reaches the envelope (negative-U 0/-2 transition).")
print(f"Bound from the 0/-2 crossing: dE(V_Al, -1; E_F=0) >= {bound:.3f} eV "
      f"(vs {FORMATION_ENERGY_AT_VALENCE_BAND_MAXIMUM[(0.333, 'N-rich, oxygen absent')]['V_Al'][0]:.3f} "
      f"eV for q = 0).")


## 10. Site densities

SI Eq. (3) is `C_D = N_sites * exp(-dE_eff / kT)`, and `N_sites` is "the concentration of the
structure sites in Al(1-x)Sc(x)N alloys where the defect can form". For the wurtzite alloy that means

- `V_N` and `O_N` on the **anion** sublattice: `N_anion = 2 / V_cell`;
- `V_Al` on the **Al** sites only: `(1 - x) * N_cation`;
- `V_Sc` on the **Sc** sites only: `x * N_cation`;

with `N_cation = N_anion` for the 1:1 wurtzite. Getting the last two right matters: at x = 0.042
there is a single Sc atom on the 24 cation sites of the supercell, so `N_sites(V_Sc)` is 24 times
smaller than the cation density, and the net carrier concentration of Fig. 2(c) moves by 1.4 decades.

Lattice constants come from the companion notebook's digitization of Hirata et al. (2020) Fig. 11,
linearly interpolated. The paper itself never tabulates its relaxed SQS cell parameters. For AlN this
gives 4.79e22 N sites/cm^3, the usual value.


In [ ]:
# Wurtzite lattice constants of Al(1-x)Sc(x)N, from Hirata et al. (2020) Fig. 11 as
# digitized in Hirata2020_AlScN_CALPHAD_v2.ipynb.
HIRATA_COMPOSITION_GRID = [0.000, 0.125, 0.250, 0.375]
HIRATA_LATTICE_PARAMETER_A = [3.1111, 3.1685, 3.2246, 3.3052]   # +-0.006 Angstrom
HIRATA_LATTICE_PARAMETER_C = [4.9832, 5.0044, 5.0044, 4.9702]   # +-0.006 Angstrom


def interpolate(grid, values, position):
    for index in range(len(grid) - 1):
        if grid[index] <= position <= grid[index + 1]:
            weight = (position - grid[index]) / (grid[index + 1] - grid[index])
            return values[index] + weight * (values[index + 1] - values[index])
    return values[-1] if position > grid[-1] else values[0]


def wurtzite_cell_volume_cubic_angstrom(composition):
    parameter_a = interpolate(HIRATA_COMPOSITION_GRID, HIRATA_LATTICE_PARAMETER_A, composition)
    parameter_c = interpolate(HIRATA_COMPOSITION_GRID, HIRATA_LATTICE_PARAMETER_C, composition)
    return (math.sqrt(3.0) / 2.0) * parameter_a * parameter_a * parameter_c


def anion_site_density_per_cubic_centimetre(composition):
    """Two formula units per wurtzite primitive cell, one N site each."""
    return 2.0 / (wurtzite_cell_volume_cubic_angstrom(composition) * 1.0e-24)


def site_density(defect, composition):
    anion = anion_site_density_per_cubic_centimetre(composition)
    scandium_fraction = SCANDIUM_ATOMS_PER_SUPERCELL[composition] / CATION_SITES_PER_SUPERCELL
    if defect == "V_Sc":
        return anion * scandium_fraction
    if defect == "V_Al":
        return anion * (1.0 - scandium_fraction)
    return anion


check_value(anion_site_density_per_cubic_centimetre(0.0), 4.79e22, 0.01,
            "AlN nitrogen site density from the Hirata lattice constants")

print(f"{'x':>6} {'a (A)':>8} {'c (A)':>8} {'V (A^3)':>9} {'N sites':>12} "
      f"{'Al sites':>12} {'Sc sites':>12}")
for composition in COMPOSITIONS:
    print(f"{composition:6.3f} "
          f"{interpolate(HIRATA_COMPOSITION_GRID, HIRATA_LATTICE_PARAMETER_A, composition):8.4f} "
          f"{interpolate(HIRATA_COMPOSITION_GRID, HIRATA_LATTICE_PARAMETER_C, composition):8.4f} "
          f"{wurtzite_cell_volume_cubic_angstrom(composition):9.3f} "
          f"{site_density('V_N', composition):12.4e} "
          f"{site_density('V_Al', composition):12.4e} "
          f"{site_density('V_Sc', composition):12.4e}")


## 11. Equilibrium defect thermodynamics

`C(D,q) = N_sites(D) * exp(-dE(D,q; E_F) / kT)`, summed over **every** charge state — which is why
section 9 mattered. Charge neutrality is `p - n + sum_D sum_q q * C(D,q) = 0`, solved for `E_F,eq` by
bisection on a function that is monotonically decreasing in `E_F`.

### The density-of-states assumption, stated plainly

The paper obtains `n` and `p` by integrating the **ab-initio** density of states with `py-sc-fermi`.
We do not have that DOS. We use parabolic bands,

    n = N_C exp(-(E_gap - E_F)/kT),    p = N_V exp(-E_F/kT),
    N_C, N_V = 2.5093e19 * (m*/m_0)^(3/2) * (T/300)^(3/2) cm^-3,

with `m_e* = 0.4` and `m_h* = 3.5` (AlN band-edge values; the alloy values are not published). **This
is an assumption, not a result**, and section 13 shows where it breaks: under most-N-poor conditions
`E_F,eq` is pinned by the defects and the masses are irrelevant to within 0.002 eV, while under
N-rich conditions the defect concentrations fall to ~1e4 cm^-3, the material is essentially
intrinsic, and `E_F,eq` is set entirely by the DOS.

One consequence worth stating up front: charge neutrality means the **net carrier concentration
`|p - n|` equals `|sum_D sum_q q * C(D,q)|` identically**, so Figure 2(c) can be reproduced from the
defect energetics alone, with no DOS assumption at all, as long as `E_F,eq` is taken from the paper.


In [ ]:
ELECTRON_EFFECTIVE_MASS = 0.4      # m_0, AlN conduction band minimum; ASSUMED for the alloy
HOLE_EFFECTIVE_MASS = 3.5          # m_0, AlN valence band density-of-states mass; ASSUMED


def band_edge_density(mass_ratio, temperature):
    return 2.5093e19 * mass_ratio ** 1.5 * (temperature / 300.0) ** 1.5


def carrier_concentrations(fermi_energy, band_gap, temperature,
                           electron_mass=ELECTRON_EFFECTIVE_MASS,
                           hole_mass=HOLE_EFFECTIVE_MASS):
    thermal_energy = BOLTZMANN_CONSTANT_ELECTRONVOLT_PER_KELVIN * temperature
    electrons = band_edge_density(electron_mass, temperature) * math.exp(
        -(band_gap - fermi_energy) / thermal_energy)
    holes = band_edge_density(hole_mass, temperature) * math.exp(-fermi_energy / thermal_energy)
    return electrons, holes


def defect_concentration(composition, condition, defect, fermi_energy, temperature):
    """Sum over ALL charge states, not just the ground state."""
    thermal_energy = BOLTZMANN_CONSTANT_ELECTRONVOLT_PER_KELVIN * temperature
    sites = site_density(defect, composition)
    charge_states = FORMATION_ENERGY_AT_VALENCE_BAND_MAXIMUM[(composition, condition)][defect]
    return sum(sites * math.exp(-(energy + charge * fermi_energy) / thermal_energy)
               for charge, energy in charge_states.items())


def net_defect_charge(composition, condition, fermi_energy, temperature):
    thermal_energy = BOLTZMANN_CONSTANT_ELECTRONVOLT_PER_KELVIN * temperature
    total = 0.0
    for defect, charge_states in FORMATION_ENERGY_AT_VALENCE_BAND_MAXIMUM[
            (composition, condition)].items():
        sites = site_density(defect, composition)
        for charge, energy in charge_states.items():
            total += charge * sites * math.exp(
                -(energy + charge * fermi_energy) / thermal_energy)
    return total


def charge_neutrality_residual(fermi_energy, composition, condition, temperature,
                               electron_mass=ELECTRON_EFFECTIVE_MASS,
                               hole_mass=HOLE_EFFECTIVE_MASS):
    electrons, holes = carrier_concentrations(fermi_energy, BAND_GAP[composition],
                                              temperature, electron_mass, hole_mass)
    return net_defect_charge(composition, condition, fermi_energy, temperature) + holes - electrons


def solve_equilibrium_fermi_energy(composition, condition, temperature,
                                   electron_mass=ELECTRON_EFFECTIVE_MASS,
                                   hole_mass=HOLE_EFFECTIVE_MASS, iterations=200):
    """Bisection; the residual decreases monotonically in E_F."""
    lower, upper = -1.0, BAND_GAP[composition] + 1.0
    for _ in range(iterations):
        middle = 0.5 * (lower + upper)
        if charge_neutrality_residual(middle, composition, condition, temperature,
                                      electron_mass, hole_mass) > 0.0:
            lower = middle
        else:
            upper = middle
    return 0.5 * (lower + upper)


print(f"{'x':>6} {'condition':<28} {'E_F solved':>11} {'E_F marker':>11} {'difference':>11}")
for composition in COMPOSITIONS:
    for condition in CONDITIONS:
        solved = solve_equilibrium_fermi_energy(composition, condition,
                                                REFERENCE_TEMPERATURE_KELVIN)
        marker = EQUILIBRIUM_FERMI_ENERGY_AT_1000_KELVIN[(composition, condition)]
        print(f"{composition:6.3f} {condition:<28} {solved:11.4f} {marker:11.4f} "
              f"{solved - marker:+11.4f}")


## 12. How much the effective masses matter

Sweeping the assumed masses over the whole plausible range for a wide-gap nitride separates the two
regimes cleanly.


In [ ]:
MASS_TRIALS = [(0.3, 2.0), (0.4, 3.5), (0.5, 5.0), (1.0, 1.0)]

print(f"{'x':>6} {'condition':<28} " +
      " ".join(f"{'me=%.1f mh=%.1f' % pair:>16}" for pair in MASS_TRIALS))
mass_spread_by_condition = {}
for composition in COMPOSITIONS:
    for condition in ("N-rich, oxygen absent", "most N-poor, oxygen absent"):
        solutions = [solve_equilibrium_fermi_energy(composition, condition,
                                                    REFERENCE_TEMPERATURE_KELVIN, *pair)
                     for pair in MASS_TRIALS]
        spread = max(solutions) - min(solutions)
        mass_spread_by_condition.setdefault(condition, []).append(spread)
        print(f"{composition:6.3f} {condition:<28} " +
              " ".join(f"{value:16.4f}" for value in solutions) + f"   spread {spread:.4f}")

check_zero(max(mass_spread_by_condition["most N-poor, oxygen absent"]), 0.005,
           "Most N-poor: E_F,eq is defect-pinned and insensitive to the assumed masses")
check_zero(max(mass_spread_by_condition["N-rich, oxygen absent"]), 0.005,
           "N-rich: E_F,eq is insensitive to the assumed masses",
           expect_failure=True,
           note="EXPECTED FAILURE. Under N-rich conditions every defect sits near 1e4 cm^-3, the "
                "alloy is intrinsic, and E_F,eq is set by the density of states alone. This is the "
                "one place the parabolic-band approximation is load-bearing.")

worst_marker_error_n_poor = max(
    abs(solve_equilibrium_fermi_energy(composition, "most N-poor, oxygen absent",
                                       REFERENCE_TEMPERATURE_KELVIN)
        - EQUILIBRIUM_FERMI_ENERGY_AT_1000_KELVIN[(composition, "most N-poor, oxygen absent")])
    for composition in COMPOSITIONS)
check_zero(worst_marker_error_n_poor, 0.012,
           "Most N-poor: solved E_F,eq reproduces the marker drawn in Figs. 1(b)/S4-S6(b)")

worst_marker_error_n_rich = max(
    abs(solve_equilibrium_fermi_energy(composition, "N-rich, oxygen absent",
                                       REFERENCE_TEMPERATURE_KELVIN)
        - EQUILIBRIUM_FERMI_ENERGY_AT_1000_KELVIN[(composition, "N-rich, oxygen absent")])
    for composition in COMPOSITIONS)
check_zero(worst_marker_error_n_rich, 0.012,
           "N-rich: solved E_F,eq reproduces the marker drawn in Figs. 1(a)/S4-S6(a)",
           expect_failure=True,
           note="EXPECTED FAILURE, worst case x = 0.125 at 0.28 eV. Matching the paper's markers "
                "would need an effective conduction-band density of ~3.5e21 cm^-3 rather than the "
                "3.9e19 cm^-3 of a parabolic band with m_e* = 0.4 - see section 14.")


## 13. Reproducing Figure 2

Figure 2 is the validation target: (a) V_N concentration, (b) O_N concentration and (c) net carrier
concentration, each against temperature from 500 to 1000 K, for all four compositions, solid lines
for most-N-poor and dotted for N-rich growth. The curves were read off the same way as Figure 1 —
vector polylines, sampled at six temperatures.

Section 12 established that our parabolic-band model can only place `E_F,eq` where the paper does
under **most-N-poor** conditions. The comparison is therefore split so that the two questions do not
contaminate each other:

- **13.1** uses our own self-consistently solved `E_F,eq(T)` over the whole temperature range, for the
  most-N-poor V_N curves, where the solve is defect-pinned and provably reproduces the paper's marker.
  This is the end-to-end test of the whole chain.
- **13.2 - 13.4** use the paper's own `E_F,eq` markers at 1000 K, which are *data* read off Figures 1
  and S4-S6 rather than an output of our model. That isolates SI Eq. (3), the extracted energies and
  the site densities from the density-of-states question.


In [ ]:
# ---------------------------------------------------------------------------
# Figure 2, log10 of the concentration in cm^-3, sampled at these temperatures.
# None marks a point below the plotted axis range. +-0.02 decades as read from Fig. 2.
# Solid curves are the most-N-poor condition, dotted curves are N-rich.
# ---------------------------------------------------------------------------
FIGURE_2_TEMPERATURES = [500, 600, 700, 800, 900, 1000]

FIGURE_2_NITROGEN_VACANCY = {
    (0.042, "most N-poor, oxygen absent"): [5.336, 8.225, 10.292, 11.845, 13.055, 14.026],
    (0.125, "most N-poor, oxygen absent"): [11.662, 13.495, 14.807, 15.793, 16.563, 17.182],
    (0.25, "most N-poor, oxygen absent"): [11.390, 13.265, 14.605, 15.611, 16.396, 17.025],
    (0.333, "most N-poor, oxygen absent"): [9.957, 12.072, 13.587, 14.728, 15.618, 16.335],
    (0.042, "N-rich, oxygen absent"): [None, None, None, None, None, None],
    (0.125, "N-rich, oxygen absent"): [None, None, None, None, None, 4.792],
    (0.25, "N-rich, oxygen absent"): [None, None, None, None, None, 3.556],
    (0.333, "N-rich, oxygen absent"): [None, None, None, None, None, 4.028],
}
FIGURE_2_OXYGEN_ON_NITROGEN = {
    (0.042, "most N-poor, oxygen present"): [8.844, 10.973, 12.506, 13.666, 14.576, 15.308],
    (0.125, "most N-poor, oxygen present"): [14.538, 15.739, 16.608, 17.267, 17.783, 18.199],
    (0.25, "most N-poor, oxygen present"): [15.964, 16.846, 17.479, 17.956, 18.331, 18.635],
    (0.333, "most N-poor, oxygen present"): [13.596, 14.897, 15.839, 16.557, 17.123, 17.584],
    (0.042, "N-rich, oxygen present"): [None, 7.993, 9.909, 11.347, 12.465, 13.361],
    (0.125, "N-rich, oxygen present"): [7.308, 9.683, 11.385, 12.664, 13.662, 14.463],
    (0.25, "N-rich, oxygen present"): [9.992, 11.921, 13.299, 14.332, 15.136, 15.780],
    (0.333, "N-rich, oxygen present"): [10.086, 12.015, 13.392, 14.425, 15.229, 15.873],
}
FIGURE_2_NET_CARRIER = {
    (0.042, "most N-poor, oxygen absent"): [None, None, 7.822, 9.582, 10.956, 12.061],
    (0.125, "most N-poor, oxygen absent"): [None, None, 7.199, 9.016, 10.436, 11.578],
    (0.25, "most N-poor, oxygen absent"): [None, 6.423, 8.622, 10.281, 11.577, 12.620],
    (0.333, "most N-poor, oxygen absent"): [5.644, 8.339, 10.274, 11.733, 12.872, 13.788],
    (0.042, "N-rich, oxygen absent"): [None, None, None, None, None, 6.447],
    (0.125, "N-rich, oxygen absent"): [None, None, None, None, None, None],
    (0.25, "N-rich, oxygen absent"): [None, None, None, None, 6.944, 8.420],
    (0.333, "N-rich, oxygen absent"): [None, None, None, None, None, 7.465],
}
print("Figure 2 digitized at", FIGURE_2_TEMPERATURES, "K")


### 13.1 V_N under most-N-poor conditions, with a self-consistently solved Fermi level

Nothing here is fitted. `E_F,eq(T)` comes out of the bisection at each temperature, the site density
comes out of the lattice constants, the formation energies come out of the figures, and SI Eq. (3)
carries no degeneracy factor.


In [ ]:
def concentration_at(composition, condition, quantity, fermi_energy, temperature):
    if quantity == "net":
        return abs(net_defect_charge(composition, condition, fermi_energy, temperature))
    return defect_concentration(composition, condition, quantity, fermi_energy, temperature)


print("Figure 2(a): V_N, most N-poor, E_F,eq(T) solved self-consistently at every temperature")
print(f"{'x':>6} {'T':>5} {'E_F solved':>11} {'computed':>9} {'plotted':>9} {'diff':>7}")
nitrogen_vacancy_errors = []
for composition in COMPOSITIONS:
    condition = "most N-poor, oxygen absent"
    for temperature, expected in zip(FIGURE_2_TEMPERATURES,
                                     FIGURE_2_NITROGEN_VACANCY[(composition, condition)]):
        if expected is None:
            continue
        fermi_energy = solve_equilibrium_fermi_energy(composition, condition, temperature)
        computed = math.log10(concentration_at(composition, condition, "V_N",
                                               fermi_energy, temperature))
        nitrogen_vacancy_errors.append(computed - expected)
        print(f"{composition:6.3f} {temperature:5d} {fermi_energy:11.4f} {computed:9.3f} "
              f"{expected:9.3f} {computed - expected:+7.3f}")

mean_offset = sum(nitrogen_vacancy_errors) / len(nitrogen_vacancy_errors)
check_zero(max(abs(value) for value in nitrogen_vacancy_errors), 0.03,
           "Figure 2(a): V_N over 500-1000 K, all four compositions, most N-poor",
           note=f"{len(nitrogen_vacancy_errors)} samples; mean offset {mean_offset:+.3f} decades "
                f"({10**mean_offset - 1:+.1%} in concentration); no adjustable parameter")


### 13.2 V_N under N-rich conditions, at the paper's own Fermi-level markers


In [ ]:
print("Figure 2(a): V_N, N-rich, at the E_F,eq markers of Figs. 1(a)/S4-S6(a)")
print(f"{'x':>6} {'E_F marker':>11} {'computed':>9} {'plotted':>9} {'diff':>7}")
n_rich_errors = []
for composition in COMPOSITIONS:
    condition = "N-rich, oxygen absent"
    expected = FIGURE_2_NITROGEN_VACANCY[(composition, condition)][-1]
    fermi_energy = EQUILIBRIUM_FERMI_ENERGY_AT_1000_KELVIN[(composition, condition)]
    computed = math.log10(concentration_at(composition, condition, "V_N", fermi_energy,
                                           REFERENCE_TEMPERATURE_KELVIN))
    if expected is None:
        print(f"{composition:6.3f} {fermi_energy:11.4f} {computed:9.3f} "
              f"{'off-scale':>9} {'--':>7}")
        continue
    n_rich_errors.append(computed - expected)
    print(f"{composition:6.3f} {fermi_energy:11.4f} {computed:9.3f} {expected:9.3f} "
          f"{computed - expected:+7.3f}")

check_zero(max(abs(value) for value in n_rich_errors), 0.07,
           "Figure 2(a): V_N at 1000 K, N-rich, using the paper's own E_F,eq markers",
           note="the paper states V_N is ~1e4 cm^-3 at 1000 K under N-rich conditions and below "
                "1e6 cm^-3 across composition; the recovered values are 1e2.5 to 1e4.8")

# The same numbers with OUR solved Fermi level instead of the paper's marker, to size
# the density-of-states problem of section 12 in decades rather than in eV.
n_rich_errors_own_fermi = []
for composition in COMPOSITIONS:
    condition = "N-rich, oxygen absent"
    expected = FIGURE_2_NITROGEN_VACANCY[(composition, condition)][-1]
    if expected is None:
        continue
    fermi_energy = solve_equilibrium_fermi_energy(composition, condition,
                                                  REFERENCE_TEMPERATURE_KELVIN)
    n_rich_errors_own_fermi.append(
        math.log10(concentration_at(composition, condition, "V_N", fermi_energy,
                                    REFERENCE_TEMPERATURE_KELVIN)) - expected)
check_zero(max(abs(value) for value in n_rich_errors_own_fermi), 0.07,
           "Figure 2(a): V_N at 1000 K, N-rich, using our own parabolic-band E_F,eq",
           expect_failure=True,
           note=f"EXPECTED FAILURE, worst case {max(abs(v) for v in n_rich_errors_own_fermi):.2f} "
                f"decades. Identical inputs except for E_F,eq: this is the cost, in concentration, "
                f"of the density-of-states approximation of section 12.")


### 13.3 O_N: the one quantity that does not reproduce


In [ ]:
print("Figure 2(b): O_N at 1000 K, at the E_F,eq markers of Figs. 1(c,d)/S4-S6(c,d)")
print(f"{'x':>6} {'condition':<28} {'computed':>9} {'plotted':>9} {'diff':>7} {'ratio':>7}")
oxygen_errors = []
for composition in COMPOSITIONS:
    for condition in ("N-rich, oxygen present", "most N-poor, oxygen present"):
        expected = FIGURE_2_OXYGEN_ON_NITROGEN[(composition, condition)][-1]
        fermi_energy = EQUILIBRIUM_FERMI_ENERGY_AT_1000_KELVIN[(composition, condition)]
        computed = math.log10(concentration_at(composition, condition, "O_N", fermi_energy,
                                               REFERENCE_TEMPERATURE_KELVIN))
        oxygen_errors.append(computed - expected)
        print(f"{composition:6.3f} {condition:<28} {computed:9.3f} {expected:9.3f} "
              f"{computed - expected:+7.3f} {10 ** (computed - expected):7.2f}")

mean_oxygen_offset = sum(oxygen_errors) / len(oxygen_errors)
oxygen_spread = max(oxygen_errors) - min(oxygen_errors)
check_zero(max(abs(value) for value in oxygen_errors), 0.07,
           "Figure 2(b): O_N from SI Eq. (3) and the Fig. 1(c,d)/S4-S6(c,d) energies",
           expect_failure=True,
           note=f"EXPECTED FAILURE. Offset {mean_oxygen_offset:+.3f} decades, i.e. a factor "
                f"{10 ** mean_oxygen_offset:.1f}. See section 14.3.")
check_zero(oxygen_spread, 0.10,
           "Figure 2(b): the O_N offset is one constant factor, not a composition or vertex trend",
           note=f"spread {oxygen_spread:.3f} decades on an offset of {mean_oxygen_offset:.3f} "
                f"over 4 compositions x 2 chemical-potential vertices; a chemical-potential error "
                f"could not do that, a site-count error would. "
                f"24 N sites per 48-atom SQS = {math.log10(24.0):.3f} decades.")


### 13.4 Net carrier concentration, from charge neutrality alone


In [ ]:
print("Figure 2(c) at 1000 K: |p - n| = |sum_D sum_q q C(D,q)|, no density of states needed")
print(f"{'x':>6} {'condition':<28} {'sign':>5} {'computed':>9} {'plotted':>9} {'diff':>7} "
      f"{'dec/2meV':>9}")
CONDITIONING_THRESHOLD_DECADES_PER_TWO_MILLIELECTRONVOLT = 0.7
robust_errors, fragile_errors, fragile_sensitivities = [], [], []
for composition in COMPOSITIONS:
    for condition in ("N-rich, oxygen absent", "most N-poor, oxygen absent"):
        expected = FIGURE_2_NET_CARRIER[(composition, condition)][-1]
        if expected is None:
            continue
        fermi_energy = EQUILIBRIUM_FERMI_ENERGY_AT_1000_KELVIN[(composition, condition)]
        signed = net_defect_charge(composition, condition, fermi_energy,
                                   REFERENCE_TEMPERATURE_KELVIN)
        nudged = net_defect_charge(composition, condition, fermi_energy + 0.002,
                                   REFERENCE_TEMPERATURE_KELVIN)
        sensitivity = abs(math.log10(abs(nudged / signed)))
        difference = math.log10(abs(signed)) - expected
        if sensitivity > CONDITIONING_THRESHOLD_DECADES_PER_TWO_MILLIELECTRONVOLT:
            fragile_errors.append(difference)
            fragile_sensitivities.append(sensitivity)
        else:
            robust_errors.append(difference)
        print(f"{composition:6.3f} {condition:<28} {'e-' if signed > 0 else 'h+':>5} "
              f"{math.log10(abs(signed)):9.3f} {expected:9.3f} {difference:+7.3f} "
              f"{sensitivity:9.2f}")

check_zero(max(abs(value) for value in robust_errors), 0.30,
           "Figure 2(c): net carrier concentration where the charge balance is well conditioned",
           note=f"{len(robust_errors)} points changing by less than "
                f"{CONDITIONING_THRESHOLD_DECADES_PER_TWO_MILLIELECTRONVOLT} decades per 2 meV "
                f"of E_F. The sign also comes out right everywhere: holes under N-rich, electrons "
                f"under most N-poor, as the paper states.")
check_zero(max(abs(value) for value in fragile_errors), 0.30,
           "Figure 2(c): net carrier concentration where the charge balance is ill conditioned",
           expect_failure=True,
           note=f"EXPECTED FAILURE. {len(fragile_errors)} points at which donors and acceptors "
                f"cancel to 5-6 decades, so up to "
                f"{max(fragile_sensitivities):.0f} decades of the answer move per 2 meV of E_F - "
                f"the sign of p - n even flips inside that window - against the +-2 meV with which "
                f"E_F,eq can be read off the figures. This is a resolution limit of the "
                f"reproduction, not a disagreement with the paper.")

print()
print("Effective conduction-band density needed to reproduce Fig. 2(c) at the paper's own")
print("most-N-poor E_F,eq markers, where E_F,eq is defect-pinned and n = |sum_D sum_q q C|:")
print(f"{'x':>6} {'E_C - E_F':>10} {'n plotted':>12} {'N_C needed':>12} "
      f"{'parabolic m*':>13} {'N_C at m*=0.4':>14}")
effective_band_edge_densities = []
for composition in COMPOSITIONS:
    condition = "most N-poor, oxygen absent"
    fermi_energy = EQUILIBRIUM_FERMI_ENERGY_AT_1000_KELVIN[(composition, condition)]
    electron_binding = BAND_GAP[composition] - fermi_energy
    plotted_electrons = 10.0 ** FIGURE_2_NET_CARRIER[(composition, condition)][-1]
    needed = plotted_electrons / math.exp(
        -electron_binding / (BOLTZMANN_CONSTANT_ELECTRONVOLT_PER_KELVIN
                             * REFERENCE_TEMPERATURE_KELVIN))
    effective_band_edge_densities.append(needed)
    equivalent_mass = (needed / band_edge_density(1.0, REFERENCE_TEMPERATURE_KELVIN)) ** (2.0 / 3.0)
    print(f"{composition:6.3f} {electron_binding:10.4f} {plotted_electrons:12.3e} "
          f"{needed:12.3e} {equivalent_mass:13.2f} "
          f"{band_edge_density(ELECTRON_EFFECTIVE_MASS, REFERENCE_TEMPERATURE_KELVIN):14.3e}")
mean_effective_density = (sum(effective_band_edge_densities)
                          / len(effective_band_edge_densities))
spread_effective_density = (max(effective_band_edge_densities)
                            - min(effective_band_edge_densities)) / 2.0
print(f"mean {mean_effective_density:.3e} +- {spread_effective_density:.3e} cm^-3, "
      f"the same for all four compositions")

check_zero(spread_effective_density / mean_effective_density, 0.30,
           "The effective conduction-band density implied by Fig. 2(c) is composition-independent",
           note=f"{mean_effective_density:.2e} +- {spread_effective_density:.2e} cm^-3 at 1000 K "
                f"against {band_edge_density(ELECTRON_EFFECTIVE_MASS, 1000.0):.2e} cm^-3 for a "
                f"parabolic band with m_e* = {ELECTRON_EFFECTIVE_MASS}. E_F,eq sits 1.6-2.0 eV "
                f"below the CBM, deep enough that the true density of states far exceeds the "
                f"sqrt(E) band-edge form - which is a property of the DOS, not of the defects.")


## 14. Findings

### 14.1 The extraction is exact

The figures are vector art, so the only error sources are the calibration (uniform tick spacings,
good to ~2e-4 eV) and the figures' own vertex rendering. Across all 244 recovered charge-state lines:

| check | result |
|---|---|
| fitted slope minus nearest integer | worst 5.6e-4, rms 1.05e-4, over 244 segments |
| kinks of the reconstructed envelope vs the drawn kinks | worst 0.031 eV, mean 0.0028 eV, over 188 kinks |
| charge transition levels vs the independently drawn Fig. 3 bars | worst 0.020 eV |
| transition levels across the four chemical-potential panels (must be identical) | worst 0.028 eV |
| band gap from panel width vs Fig. 3 bars | worst 0.0001 eV |
| N-poor minus N-rich energy vs Delta-mu of SI Tables S5 / S9 | worst 0.004 eV |

The last row deserves emphasis: the *difference* between the N-rich and most-N-poor panels reproduces
Delta-mu_Al, Delta-mu_Sc, Delta-mu_N and Delta-mu_N - Delta-mu_O of the supplementary tables to
0.004 eV, and those tables were not used anywhere in the extraction.

**Quoted uncertainty on the formation energies: +-0.02 eV.**

### 14.2 Figure 2(a) is reproduced with no adjustable parameter

V_N concentration under most-N-poor conditions, four compositions, six temperatures from 500 to
1000 K, with `E_F,eq(T)` solved self-consistently at each temperature: agreement to
**+0.004 to +0.019 decades**, i.e. 1-4 % in concentration, with a mean offset of +0.012 decades.
Nothing was fitted. Under N-rich conditions at 1000 K, at the paper's own `E_F,eq` markers, the
agreement is +0.012 to +0.058 decades. The small positive bias is consistent with the +-0.006
Angstrom uncertainty on the interpolated lattice constants (0.006 decades) plus the finite stroke
width of the plotted curve.

The solved `E_F,eq(T)` is also *constant in temperature* to within 0.0001 eV and equals the paper's
1000 K marker to 0.000-0.010 eV, which is what "defect-pinned" means quantitatively.

This validates, simultaneously: the extracted `dE(q, E_F=0)`, the site density, SI Eq. (3) **without a
degeneracy factor**, the summation over all charge states, and the charge-neutrality solver.

### 14.3 Figure 2(b) is a factor of ~24 off, and the factor is a constant

Applying the paper's own Eq. (3) to the paper's own O_N energies gives concentrations
**24.2 +- 1.4 times higher** than Figure 2(b) plots. The offset is `+1.383` decades with a spread of
only **0.076 decades** across 4 compositions x 2 chemical-potential vertices - and
`log10(24) = 1.380`.

Candidate explanations, and what the constancy rules out:

- **A chemical-potential difference** would have to be the same at the V1 and V4 vertices of four
  different quaternary stability regions. Tables S6-S9 contain no such common offset. Ruled out.
- **The SI's temperature-dependent oxygen chemical potential** (SI Eq. 5) is worth -1.13 eV at
  1000 K, not the +0.275 eV that would be needed, and it varies with temperature while the observed
  offset does not. Ruled out.
- **A site-count normalisation** of one site per 48-atom SQS cell instead of the 24 N sites it
  contains gives exactly 24. Correcting the +0.013-decade systematic of section 14.2 puts the
  measured factor at 23.5-24.2. This is the only candidate that is naturally composition-,
  vertex- and temperature-independent, and it is numerically right.

Stated neutrally: **Figure 2(b) is not reproducible from Figures 1(c,d) and S4-S6(c,d) via SI
Eq. (3); the two differ by a constant factor of 24.2 +- 1.4, equivalently by a uniform
+0.275 +- 0.006 eV in dE(O_N) at 1000 K.** For downstream use, the Figure 1 energies are the ones cross-checked
against Figure 3 and the SI chemical-potential tables, so they are the ones this notebook keeps.

### 14.4 Figure 2(c) splits into a well-conditioned and an ill-conditioned half

Charge neutrality makes `|p - n|` identically equal to `|sum_D sum_q q C(D,q)|`, so Figure 2(c) needs
no DOS. Where the donor and acceptor sums do not cancel catastrophically the agreement is
**0.012 to 0.211 decades**. Where they cancel to 5-6 decades (x = 0.125 and 0.250 under most-N-poor
conditions) the result changes by 6-13 decades per 2 meV of `E_F,eq`, and even its **sign** flips inside that
window - while `E_F,eq` can only be read off the figures to about +-2 meV. Those points are not a disagreement with the paper; they are
below the resolution of the reproduction.

Getting the **site multiplicity** right is what makes the well-conditioned half work: with
`N_sites(V_Sc) = N_cation` instead of `x * N_cation`, the x = 0.042 N-rich point is 1.39 decades
wrong; with `x * N_cation` it is 0.012 decades wrong.

### 14.5 The parabolic-band approximation fails exactly where it is load-bearing

Under most-N-poor conditions `E_F,eq` is defect-pinned: our solved value matches the paper's marker to
**0.000-0.010 eV** and moves by less than 0.002 eV as `m_e*` goes from 0.3 to 1.0 and `m_h*` from 1.0
to 5.0. Under N-rich conditions every defect sits near 1e4 cm^-3, the alloy is intrinsic, and
`E_F,eq` is set by the DOS alone: our solved value misses the marker by up to **0.28 eV**
(worst case x = 0.125).

The size of the gap is informative. To place `E_F,eq` where the paper does under most-N-poor
conditions **and** reproduce Figure 2(c)'s absolute electron concentration would require an effective
conduction-band density of `3.5e21 +- 0.9e21 cm^-3` at 1000 K - the same value for all four
compositions - against `3.9e19 cm^-3` for a parabolic band with `m_e* = 0.4`. That is a factor of
~90, or a parabolic-equivalent mass of ~8 m_0. It is not a sign that the defect energetics are wrong:
`E_F,eq` sits 1.6-2.0 eV *below* the conduction band minimum, deep enough that the true DOS is far
above the sqrt(E) band-edge form, and integrating it is precisely what `py-sc-fermi` does and we
cannot.

### 14.6 What could not be extracted, and why

| item | status |
|---|---|
| `V_Al` q = -1 at x = 0.333 | never reaches the envelope (negative-U 0/-2); only a lower bound, computed in section 9 |
| `V_N` q = +3 under most-N-poor conditions | its +3/+2 level lies below E_F = 0, so no segment is drawn |
| `O_N` q = -1 at x = 0.042 and 0.125 | its 0/-1 level lies above the CBM; consistent with Fig. 3 drawing only one O_N bar at those compositions |
| the ab-initio density of states | never published; the reason for section 14.5 |
| `dE_min` and `dE_max` separately | only the Zhang-averaged `dE_eff` is plotted, so the site-to-site spread within the alloy cannot be recovered |
| Figures S1-S3, S7-S12 | phase diagrams, local structures and switching pathways: not needed here |
| Fig. S6(b) x calibration | its tick marks are mis-emitted; recovered from integer slopes instead (section 4) |


## 15. Plots (optional - needs matplotlib)


In [ ]:
try:
    import matplotlib.pyplot as plt
    HAVE_MATPLOTLIB = True
except ImportError:
    HAVE_MATPLOTLIB = False
    print("matplotlib not installed -- skipping plots. "
          "The numerical results above are unaffected.")

DEFECT_COLOURS = {"V_Al": "#3776b8", "V_Sc": "#984ea3", "V_N": "#4daf4a", "O_N": "#e41a1c"}

PANEL_CONDITIONS = ("N-rich, oxygen absent", "most N-poor, oxygen absent",
                    "N-rich, oxygen present", "most N-poor, oxygen present")

SOURCE_FIGURE = {0.333: "Fig. 1 (main paper)", 0.042: "Fig. S4 (supplement)",
                 0.125: "Fig. S5 (supplement)", 0.250: "Fig. S6 (supplement)"}


def plot_formation_energy_panels(composition, axis_row=None):
    """One row of four panels (a)-(d) for a single composition, as laid out in the papers."""
    gap = BAND_GAP[composition]
    created = axis_row is None
    if created:
        _, axis_row = plt.subplots(1, 4, figsize=(19, 4.2), sharey=True)
    for axis, condition in zip(axis_row, PANEL_CONDITIONS):
        panel = FORMATION_ENERGY_AT_VALENCE_BAND_MAXIMUM.get((composition, condition))
        if panel is None:
            axis.set_visible(False)
            continue
        for defect in DEFECTS:
            if defect not in panel:
                continue
            fermi_energies, lines = charge_state_lines(composition, condition, defect)
            for charge, values in sorted(lines.items(), reverse=True):
                axis.plot(fermi_energies, values, color=DEFECT_COLOURS[defect],
                          alpha=0.35, linewidth=0.9)
            envelope = [lower_envelope_energy(panel[defect], value) for value in fermi_energies]
            axis.plot(fermi_energies, envelope, color=DEFECT_COLOURS[defect],
                      linewidth=2.4, label=defect)
        marker = EQUILIBRIUM_FERMI_ENERGY_AT_1000_KELVIN.get((composition, condition))
        if marker is not None:
            axis.axvline(marker, color="0.35", linestyle="--", linewidth=1.0)
        axis.set_xlim(0, gap)
        axis.set_ylim(-1, 9)
        axis.set_xlabel("E_F  [eV above the VBM]")
        axis.set_title(f"x = {composition},  {condition}\ngap = {gap:.3f} eV", fontsize=9)
        axis.grid(alpha=0.3)
        axis.legend(fontsize=7.5, loc="upper right")
    axis_row[0].set_ylabel("dE(D,q)  [eV]")
    return axis_row


if HAVE_MATPLOTLIB:
    # One figure per composition, mirroring the papers' own layout: each source figure
    # carries the four (a)-(d) chemical-potential panels for a single composition.
    for composition in sorted(BAND_GAP):
        figure, axis_row = plt.subplots(1, 4, figsize=(19, 4.2), sharey=True)
        plot_formation_energy_panels(composition, axis_row)
        figure.suptitle(
            f"Al({1-composition:.3f})Sc({composition:.3f})N   --   {SOURCE_FIGURE[composition]}"
            "        Thick: the drawn envelope.  Thin: every charge state extended as a full "
            "line (section 9).  Dashed: E_F,eq at 1000 K.", fontsize=9)
        plt.tight_layout()
        plt.show()

In [ ]:
if HAVE_MATPLOTLIB:
    # Compact overview: the lower envelopes for every composition on one axis per condition,
    # so the composition trend is visible directly.
    figure, axes = plt.subplots(1, 4, figsize=(19, 4.2), sharey=True)
    composition_styles = {0.042: ":", 0.125: "-.", 0.250: "--", 0.333: "-"}
    for axis, condition in zip(axes, PANEL_CONDITIONS):
        for composition in sorted(BAND_GAP):
            panel = FORMATION_ENERGY_AT_VALENCE_BAND_MAXIMUM.get((composition, condition))
            if panel is None:
                continue
            for defect in DEFECTS:
                if defect not in panel:
                    continue
                fermi_energies, _ = charge_state_lines(composition, condition, defect)
                envelope = [lower_envelope_energy(panel[defect], value)
                            for value in fermi_energies]
                axis.plot(fermi_energies, envelope, composition_styles[composition],
                          color=DEFECT_COLOURS[defect], linewidth=1.5,
                          label=f"{defect} x={composition}")
        axis.set_xlim(0, max(BAND_GAP.values()))
        axis.set_ylim(-1, 9)
        axis.set_xlabel("E_F  [eV above the VBM]")
        axis.set_title(condition, fontsize=9)
        axis.grid(alpha=0.3)
    axes[0].set_ylabel("dE(D,q)  [eV]")
    axes[-1].legend(fontsize=6, ncol=2, loc="upper right")
    figure.suptitle("Lower envelopes, all four compositions overlaid "
                    "(line style = composition, colour = defect)", fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
if HAVE_MATPLOTLIB:
    figure, axes = plt.subplots(1, 3, figsize=(15, 4.6))
    composition_colours = {0.042: "#3a6db8", "0.042": "#3a6db8", 0.125: "#128f75",
                           0.25: "#ee876b", 0.333: "#cc3333"}
    panels = [(FIGURE_2_NITROGEN_VACANCY, "V_N", "Fig. 2(a): V_N concentration"),
              (FIGURE_2_OXYGEN_ON_NITROGEN, "O_N", "Fig. 2(b): O_N concentration"),
              (FIGURE_2_NET_CARRIER, "net", "Fig. 2(c): net carrier concentration")]
    for axis, (table, quantity, title) in zip(axes, panels):
        for (composition, condition), plotted in sorted(table.items()):
            if all(value is None for value in plotted):
                continue
            style = "-" if condition.startswith("most") else ":"
            temperatures = [t for t, v in zip(FIGURE_2_TEMPERATURES, plotted) if v is not None]
            values = [v for v in plotted if v is not None]
            axis.plot(temperatures, values, style, color=composition_colours[composition],
                      marker="o", markersize=4, linewidth=1.2,
                      label=f"x={composition} {'N-poor' if style == '-' else 'N-rich'}")
            # Grey line: section 13.1, self-consistent E_F,eq(T), only where it is defect-pinned.
            if quantity == "V_N" and condition.startswith("most"):
                recomputed = [math.log10(concentration_at(
                    composition, condition, quantity,
                    solve_equilibrium_fermi_energy(composition, condition, temperature),
                    temperature)) for temperature in temperatures]
                axis.plot(temperatures, recomputed, "-", color="0.2", linewidth=1.0, alpha=0.8)
            # Grey cross: sections 13.2-13.4, at the paper's own 1000 K E_F,eq marker.
            fermi_energy = EQUILIBRIUM_FERMI_ENERGY_AT_1000_KELVIN[(composition, condition)]
            axis.plot([REFERENCE_TEMPERATURE_KELVIN],
                      [math.log10(concentration_at(composition, condition, quantity,
                                                   fermi_energy,
                                                   REFERENCE_TEMPERATURE_KELVIN))],
                      "x", color="0.15", markersize=9, markeredgewidth=1.6)
        axis.set_xlabel("Temperature [K]")
        axis.set_ylabel("log10 concentration [cm^-3]")
        axis.set_title(title, fontsize=10)
        axis.grid(alpha=0.3)
        axis.legend(fontsize=6, ncol=2)
    axes[1].annotate("constant factor ~24", xy=(620, 12.3), fontsize=9, color="0.15")
    figure.suptitle("Coloured: read off the published Fig. 2.  Grey crosses: recomputed here at the "
                    "paper's own 1000 K E_F,eq marker (sections 13.2-13.4).  Grey lines: recomputed "
                    "with a self-consistent E_F,eq(T) where it is defect-pinned (section 13.1).",
                    fontsize=9)
    plt.tight_layout()
    plt.show()


## 16. Forward look - what this machinery does NOT capture

*This section states a limitation and leaves a hook. No model is developed here.*

Everything above is **bulk equilibrium**. It answers "what defect population would Al(1-x)Sc(x)N carry
if it were held at temperature T until it stopped changing, in contact with reservoirs at the
tabulated chemical potentials". That is the right question for the paper's purpose and the wrong
question for a **sputtered film**, for three reasons that compound.

**1. The equilibrium answer is too small to matter.** Section 13 reproduces the paper: even at the
most N-poor chemical potential that does not decompose the nitride, `[V_N]` at 1000 K is ~1e16-1e17
cm^-3, and at a realistic 400 C growth temperature it is far lower still. Reported V_N levels in
sputtered AlScN are orders of magnitude above that. Whatever sputtered films carry, **it is not
equilibrium V_N at any accessible mu_N**.

**2. Bulk V_N cannot move at growth temperature, so nothing equilibrates after burial.** The
climbing-image NEB migration barrier for V_N in Al(1-x)Sc(x)N is **2.46 eV at -4 % strain to 2.71 eV
at +4 %** (*ACS Appl. Mater. Interfaces* **16** (2024), doi:10.1021/acsami.4c03442). With an attempt
frequency of 1e13 s^-1 that is under **one hop per vacancy over an entire ten-minute deposition** at
every usable growth temperature - roughly 2e-3 hops at 400 C, 4e-2 at 450 C. There is no bulk
equilibration and no freeze-out *depth* in the diffusion sense: the configuration present at the
instant of burial is the configuration that survives, permanently.

**3. The relevant formation energy is therefore the sub-surface one, not the bulk one.** With a
residence time per monolayer of ~0.75 s at 20 nm/min, the top one to two monolayers do locally
equilibrate if the surface barrier is <~1.5 eV, while everything below is frozen absolutely; the
transition is only ~1-2 monolayers wide because the rate is exponential in the barrier. So the
quantity a growth model needs is `dH(V_N)` **as a function of depth below the growth front**, at the
freeze-out plane, for both Al-polar and N-polar (0001) - a slab calculation, not the bulk supercell
calculation reproduced here.

**The hook.** The machinery in sections 9-13 is reusable as-is for that model: replace
`FORMATION_ENERGY_AT_VALENCE_BAND_MAXIMUM` with depth-resolved sub-surface values, replace the
tabulated `Delta-mu_N` with an effective `mu_N` set by the plasma, and solve the same charge
neutrality locally at the freeze-out plane and growth temperature. What must be added on top and is
absent here: an **athermal** channel for displacement damage from energetic-neutral and negative-ion
bombardment, and the strain dependence of `dH(V_N)` (formation energy falls under compression), since
N2 flow moves the nitrogen supply and the film stress state through the same knob. See
`agents/workdir/reusable/scratch_VN_kinetics_modeling.md`.


## 17. Check summary


In [ ]:
summarise_checks()
